# Voxa Validation: Full Architecture & OOD Testing

This notebook tests the **full Voxa pipeline** (Silero VAD + YAMNet + OOD Gates). It specifically tests if the system can reject:
1. The same person saying a different word.
2. A completely different person saying the enrolled words.

In [22]:
!pip install -q torchaudio soundfile sounddevice
import os, glob, random, tarfile, urllib.request
from collections import defaultdict
import numpy as np
import torch, torchaudio
import tensorflow_hub as hub
from scipy.spatial.distance import cosine

print('Loading YAMNet...')
yamnet = hub.load('https://tfhub.dev/google/yamnet/1')

print('Loading Silero VAD...')
vad_model, utils = torch.hub.load(repo_or_dir='snakers4/silero-vad',
                                  model='silero_vad',
                                  force_reload=False,
                                  trust_repo=True)
(get_speech_timestamps, _, read_audio, *_) = utils
print('Models loaded successfully.')

Loading YAMNet...
Loading Silero VAD...
Models loaded successfully.


Using cache found in /root/.cache/torch/hub/snakers4_silero-vad_master


## 1. The Full Preprocessing Pipeline (VAD + Extract)

In [23]:
def load_and_vad_pad(path, sr=16000, pad_sec=1.44):
    try:
        wav, orig_sr = torchaudio.load(path)
    except:
        return None
    if wav.shape[0] > 1:
        wav = wav.mean(dim=0, keepdim=True)
    if orig_sr != sr:
        wav = torchaudio.transforms.Resample(orig_sr, sr)(wav)
    
    audio = wav.squeeze(0)
    timestamps = get_speech_timestamps(audio, vad_model, sampling_rate=sr)
    if not timestamps:
        return None
        
    start_idx = timestamps[0]['start']
    end_idx = timestamps[-1]['end']
    speech = audio[start_idx:end_idx].numpy()
    
    n = int(pad_sec * sr)
    
    # FIX: Use repeat padding instead of zeros to prevent embedding degradation
    if len(speech) < n:
        repeats = int(np.ceil(n / len(speech)))
        speech = np.tile(speech, repeats)
        
    # Center crop to exactly 1.44s
    c = len(speech) // 2
    h = n // 2
    return speech[c - h : c + h].astype(np.float32)

def extract_2048d(audio):
    """Extract YAMNet embeddings and concatenate to 2048-D, L2-normalized."""
    _, emb, _ = yamnet(audio)
    emb = emb.numpy()
    if emb.shape[0] < 2:
        pad = np.zeros((2, 1024), dtype=np.float32)
        pad[:emb.shape[0]] = emb
        emb = pad
    vec = np.concatenate([emb[0], emb[1]])
    norm = np.linalg.norm(vec)
    return vec / norm if norm > 0 else vec

## 2. Enrollment & OOD Classifier

In [24]:
def qc_outlier_rejection(vecs, sigma=2.0):
    vecs = np.array(vecs)
    if len(vecs) < 3: return vecs
    c = vecs.mean(axis=0)
    dists = np.array([cosine(v, c) for v in vecs])
    return vecs[dists <= dists.mean() + sigma * dists.std()]

def enroll(paths, n_enroll=10):
    random.shuffle(paths)
    enroll_paths = paths[:n_enroll]
    test_paths   = paths[n_enroll:]

    vecs = []
    for p in enroll_paths:
        a = load_and_vad_pad(p)
        if a is not None:
            vecs.append(extract_2048d(a))

    if not vecs: return None, test_paths
    valid = qc_outlier_rejection(vecs)
    if len(valid) == 0: return None, test_paths

    centroid = valid.mean(axis=0)
    norm = np.linalg.norm(centroid)
    centroid = centroid / norm if norm > 0 else centroid
    return centroid, test_paths

def classify(vec, centroids_map, ood_thresholds_map, margin_threshold=0.05):
    scores = {name: 1.0 - cosine(vec, c) for name, c in centroids_map.items()}
    ranked = sorted(scores.items(), key=lambda x: x[1], reverse=True)
    best, best_score = ranked[0]
    
    # GATE 1: PER-INTENT OOD Threshold
    # Look up the specific threshold for the best-matching intent
    intent_threshold = ood_thresholds_map.get(best, 0.90)
    if best_score < intent_threshold:
        return 'REJECT_OOD', best_score
        
    # GATE 2: Margin Gate
    if len(ranked) > 1:
        second_score = ranked[1][1]
        margin = best_score - second_score
        if margin < margin_threshold:
            return 'REJECT_MARGIN', best_score
            
    return best, best_score

## 3. Dataset Download & Indexing

In [25]:
SC_URL = 'http://download.tensorflow.org/data/speech_commands_v0.02.tar.gz'
SC_TAR = 'speech_commands.tar.gz'
VS_DIR = 'SpeechCommands'

if not os.path.exists(VS_DIR):
    print('Downloading Speech Commands (~2.3GB)...')
    urllib.request.urlretrieve(SC_URL, SC_TAR)
    print('Extracting...')
    with tarfile.open(SC_TAR, 'r:gz') as tar:
        tar.extractall(path=VS_DIR)
    os.remove(SC_TAR)
    print('Done.')

speaker_index = defaultdict(lambda: defaultdict(list))
for wav_path in glob.glob(os.path.join(VS_DIR, '**', '*.wav'), recursive=True):
    if '_background_noise_' in wav_path: continue
    sound_class = os.path.basename(os.path.dirname(wav_path))
    fname = os.path.basename(wav_path)
    parts = fname.split('_')
    if len(parts) < 1: continue
    speaker_id = parts[0]
    speaker_index[speaker_id][sound_class].append(wav_path)

# Find speakers with at least 15 'yes', 15 'no', and 10 of some other word ('stop')
eligible = []
for pid, classes in speaker_index.items():
    if len(classes.get('yes', [])) >= 15 and len(classes.get('no', [])) >= 15 and len(classes.get('stop', [])) >= 10:
        eligible.append(pid)

print(f'Found {len(eligible)} eligible speakers with enough data for "yes", "no", and "stop".')

Found 1 eligible speakers with enough data for "yes", "no", and "stop".


## 4. The Rigorous Impostor Test

In [26]:
# We pick ONE speaker to enroll.
random.seed(42)
test_speaker = random.choice(eligible)
other_eligible = [p for p, c in speaker_index.items() if p != test_speaker and len(c.get('yes', [])) > 0 and len(c.get('no', [])) > 0 and len(c.get('stop', [])) > 0]
other_speaker = other_eligible[0]

print(f"═══ Enrolling Speaker {test_speaker} ═══════════════")
paths_yes = speaker_index[test_speaker]['yes']
paths_no  = speaker_index[test_speaker]['no']

centroid_yes, test_yes = enroll(paths_yes, n_enroll=10)
centroid_no, test_no   = enroll(paths_no, n_enroll=10)
centroids = {'yes': centroid_yes, 'no': centroid_no}
print("Successfully created centroids for 'yes' and 'no'.\n")

# 1. Calibrate Threshold Dynamically PER INTENT!
print("--- Calibrating Per-Intent OOD Thresholds ---")
noise_paths = glob.glob(os.path.join(VS_DIR, '_background_noise_', '*.wav'))

# FIX: Silero VAD deletes pure noise. We must bypass VAD for noise files.
def extract_raw_noise(path, sr=16000, pad_sec=1.44):
    try:
        wav, orig_sr = torchaudio.load(path)
        if orig_sr != sr: wav = torchaudio.transforms.Resample(orig_sr, sr)(wav)
        audio = wav.squeeze(0).numpy()
        n = int(pad_sec * sr)
        if len(audio) < n:
            audio = np.tile(audio, int(np.ceil(n / len(audio))))
        return audio[:n].astype(np.float32)
    except:
        return None

ood_thresholds = {}
for intent_name, centroid in centroids.items():
    noise_sims = []
    for n_path in noise_paths:
        audio = extract_raw_noise(n_path)
        if audio is not None:
            vec = extract_2048d(audio)
            # Score noise against THIS specific centroid
            noise_sims.append(1.0 - cosine(vec, centroid))
            
    if noise_sims:
        noise_mean = np.mean(noise_sims)
        noise_std = np.std(noise_sims)
        # Threshold for this intent = Noise Mean + 2*Std
        thresh = noise_mean + (2.0 * noise_std)
        # Cap minimum at 0.82 to prevent it from being too loose
        ood_thresholds[intent_name] = max(0.82, thresh)
    else:
        ood_thresholds[intent_name] = 0.88

margin_threshold = 0.04 

print(f"Per-Intent Thresholds:")
for name, t in ood_thresholds.items():
    print(f"  -> {name}: {t:.4f}")
print(f"Using strict Margin Threshold: {margin_threshold:.4f}\n")

def test_clips(paths, label_desc, expected_result):
    print(f"\nTesting: {label_desc} (Expected: {expected_result})")
    results = defaultdict(int)
    total = 0
    for p in paths[:10]: # test up to 10 clips
        a = load_and_vad_pad(p)
        if a is None: continue
        # Pass the thresholds MAP to the classifier
        pred, score = classify(extract_2048d(a), centroids, ood_thresholds, margin_threshold)
        results[pred] += 1
        total += 1
    
    for pred, count in results.items():
        print(f"  -> {pred}: {count}/{total} ({(count/total)*100:.1f}%)")

print("═══ 1. VALID MATCHES ═════════════════════════════")
test_clips(test_yes, f"Same speaker ({test_speaker}) saying 'yes'", "yes")
test_clips(test_no,  f"Same speaker ({test_speaker}) saying 'no'", "no")

print("\n═══ 2. IMPOSTOR WORD (Same Speaker) ══════════════")
test_clips(speaker_index[test_speaker]['stop'], f"Same speaker ({test_speaker}) saying 'stop'", "REJECT_OOD / REJECT_MARGIN")

print("\n═══ 3. IMPOSTOR SPEAKER (Different Person) ═══════")
test_clips(speaker_index[other_speaker]['yes'],  f"Other speaker ({other_speaker}) saying 'yes'", "REJECT_OOD / REJECT_MARGIN")
test_clips(speaker_index[other_speaker]['no'],   f"Other speaker ({other_speaker}) saying 'no'", "REJECT_OOD / REJECT_MARGIN")
test_clips(speaker_index[other_speaker]['stop'], f"Other speaker ({other_speaker}) saying 'stop'", "REJECT_OOD / REJECT_MARGIN")


═══ Enrolling Speaker c50f55b8 ═══════════════
Successfully created centroids for 'yes' and 'no'.

--- Calibrating Per-Intent OOD Thresholds ---
Per-Intent Thresholds:
  -> yes: 0.8200
  -> no: 0.8200
Using strict Margin Threshold: 0.0400

═══ 1. VALID MATCHES ═════════════════════════════

Testing: Same speaker (c50f55b8) saying 'yes' (Expected: yes)
  -> yes: 10/10 (100.0%)

Testing: Same speaker (c50f55b8) saying 'no' (Expected: no)
  -> yes: 4/8 (50.0%)
  -> no: 4/8 (50.0%)

═══ 2. IMPOSTOR WORD (Same Speaker) ══════════════

Testing: Same speaker (c50f55b8) saying 'stop' (Expected: REJECT_OOD / REJECT_MARGIN)
  -> REJECT_OOD: 9/10 (90.0%)
  -> no: 1/10 (10.0%)

═══ 3. IMPOSTOR SPEAKER (Different Person) ═══════

Testing: Other speaker (b5cf6ea8) saying 'yes' (Expected: REJECT_OOD / REJECT_MARGIN)
  -> REJECT_OOD: 6/8 (75.0%)
  -> yes: 2/8 (25.0%)

Testing: Other speaker (b5cf6ea8) saying 'no' (Expected: REJECT_OOD / REJECT_MARGIN)
  -> REJECT_OOD: 9/9 (100.0%)

Testing: Other spea